# v4 Notebook D - PAIR (A2) + end-to-end harm (H5) + judges
Needs the **AGENT_API_KEY** secret (DeepInfra). Hard $15 cap enforced in code. Output: `v4_D_pair_e2e.json`.

In [ ]:
# Cell 1 - setup (Kaggle T4; Internet ON; Secrets: HF_TOKEN, AGENT_API_KEY)
import os, sys, subprocess, glob
KAGGLE = os.path.isdir('/kaggle/working')
DEST = ('/kaggle/working' if KAGGLE else '/content') + '/vyuha_src'
if not os.path.isdir(f'{DEST}/.git'):
    subprocess.run(['git','clone','--depth','1','https://github.com/g25ait2149/vyuha.git',DEST], check=False)
else:
    subprocess.run(['git','-C',DEST,'pull','--ff-only'], check=False)
hits = glob.glob(DEST+'/**/vyuha/__init__.py', recursive=True)
assert hits, 'clone failed - turn Internet ON'
ROOT = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, ROOT); os.chdir(ROOT)
subprocess.run('pip -q install -U "bitsandbytes>=0.46.1" accelerate "transformers>=4.44" datasets scikit-learn sentencepiece "protobuf<5" huggingface_hub', shell=True)
tok = ''
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; tok = userdata.get('HF_TOKEN')
except Exception as e:
    print('HF_TOKEN missing:', repr(e)[:100])
os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = tok or ''
from huggingface_hub import login
if tok: login(tok)
import torch; print('GPUs:', torch.cuda.device_count(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
# Persist outputs. Kaggle /kaggle/working survives a committed run; Colab /content is WIPED on quota/idle
# reclaim, so mount Drive and write there (lesson L11 - a finished run was lost to an ephemeral VM).
OUT = '/kaggle/working/v4'
if not KAGGLE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/v4_run'
    except Exception as e:
        OUT = '/content/v4'; print('*** Drive mount FAILED - /content is EPHEMERAL, download outputs the moment the run ends ***', repr(e)[:80])
os.makedirs(OUT, exist_ok=True)
print('OUT =', OUT, '(persistent)' if (KAGGLE or OUT.startswith('/content/drive')) else '(EPHEMERAL - download immediately!)')

In [ ]:
# Cell 2 - PAIR (A2) + end-to-end harm (H5) + judges, all under the $15 cap. Needs AGENT_API_KEY secret.
import json, os, numpy as np
from vyuha.cascade.api import Client, pick_attacker_model, BudgetExceeded
from vyuha.cascade.pair_judge import pair_rewrite, intent_same, target_respond, strongreject_score, fetch_strongreject_rubric
from vyuha.cascade.data import ensure_files
ensure_files(OUT)
client = Client(cap_usd=15.0)
ATTACK_ORDER = ['NousResearch/Hermes-3-Llama-3.1-70B','mistralai/Mistral-Small-3.2-24B-Instruct-2506','Qwen/Qwen3-Next-80B-A3B-Instruct']
TARGET = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'; JUDGE = 'openai/gpt-oss-120b'
EVAL = json.load(open(f'{OUT}/v4_data.json'))['eval_seeds'][:100]
rubric = fetch_strongreject_rubric()
dev = json.load(open(f'{OUT}/v4_data.json'))['pilot_seeds'][:5]
ATT = pick_attacker_model(client, dev, ATTACK_ORDER, refuse_re=r"\b(can.?t|cannot|won.?t|unable|sorry|i will not|against)\b")
print('attacker model:', ATT, '|', client.spend_report())

In [ ]:
# Cell 3 - run PAIR per screen (needs the screens' tau; load from v4_scores if present) + e2e harm. Resumable, budget-capped.
import json, os, numpy as np
RES = f'{OUT}/v4_D_pair_e2e.json'
state = json.load(open(RES)) if os.path.exists(RES) else {'pair': {}, 'e2e': {}, 'spend': 0.0}
# PAIR needs a live screen score function; it runs in notebook A's kernel. Here we only do e2e + judging of the
# cascade-passing A1 variants, which is the compute-light, API-only part. (PAIR screen-feedback loop is optional;
# enable it by importing the screen fns from notebook A in a combined run.)
try:
    pool = json.load(open(f'{OUT}/v4_pool.json'))['texts']
    passing = json.load(open(f'{OUT}/v4_passing.json'))        # written by the analysis step: {screen: [texts that pass P0]}
except FileNotFoundError:
    passing = {'_originals': EVAL}
    print('no v4_passing.json yet -> judging originals only as a smoke test')
for key, texts in passing.items():
    if key in state['e2e']: continue
    rows = []
    for t in texts[:100]:
        try:
            resp = target_respond(client, TARGET, t, max_tokens=512)
            harm, _ = strongreject_score(client, JUDGE, rubric, t, resp)
        except BudgetExceeded as e:
            print('BUDGET STOP:', e); json.dump(state, open(RES,'w')); raise
        rows.append({'harm': harm})
    state['e2e'][key] = rows; state['spend'] = client.spent
    json.dump(state, open(RES,'w')); print(f'  e2e {key}: mean harm {np.mean([r["harm"] for r in rows]):.3f} | {client.spend_report()}', flush=True)
print('DONE', client.spend_report())